In [ ]:
# __ROOTBOOT__ ensure project root on sys.path (auto-added; safe to keep)
import os as _os, sys as _sys
_r = _os.path.abspath('')
while _r != _os.path.dirname(_r) and not _os.path.exists(_os.path.join(_r, '.project_root')):
    _r = _os.path.dirname(_r)
if _os.path.exists(_os.path.join(_r, '.project_root')) and _r not in _sys.path:
    _sys.path.insert(0, _r)


# Congress Trade-for-Trade D252 — Portfolio B Backtest

**Winner strategy** confirmed in `V2_03_trade_for_trade.ipynb` §7–8.  
Do not modify this notebook for exploration — use V2_03 for that.

**Setup:**
- Universe: Congressional **Purchase** disclosures from active politicians  
  (≥ `MIN_TRANSACTIONS` completed pairs, ≥ `MIN_YEARS_ACTIVE` years, ≥ `MIN_WIN_RATE` win rate)
- Entry: Quiver upload date (capped at `QUIVER_DELAY_CAP` days from SEC filing), close price
- Exit: Politician's next Sale disclosure on Quiver, OR `HOLDING_CAP_DAYS` calendar days (whichever first)
- Allocation: winner from V2_03 §8E — set `ALLOCATION_METHOD` in config below

**Confirmed results (V2_03, 2016–2025):** ~12% annualized with fixed allocation on filtered set.

> ⚠️ Exit timing depends on politicians filing their Sale disclosures on Quiver. In live trading, hold until the Quiver Sale alert fires or the cap is hit.

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, "../../")  # algo_trading/shared/

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from _shared.significance import full_significance_report, print_significance_report

In [ ]:
# =============================================================================
# CONFIGURATION — winner from V2_03 §8E
# =============================================================================
DATA_PKL            = r"C:\Users\danie\Desktop\INVESTMENT PROJECT\Congress Things\_ARCHIVE\final_data_MEGA.pkl"
QUIVER_DELAY_CAP    = 7       # max days from SEC filing to Quiver upload
HOLDING_CAP_DAYS    = 180     # max holding period in calendar days (D180 winner from capital efficiency analysis)
MIN_HOLD_DAYS       = 30      # minimum hold even if politician sells earlier
INITIAL_CAPITAL     = 10_000
MIN_TRANSACTIONS    = 15
MIN_YEARS_ACTIVE    = 4
MIN_WIN_RATE        = 0.50
PURCHASES_ONLY      = True

# ── WINNER ALLOCATION — E. 3% of total NAV, min 30-day hold ──────────────────
# V2_03 §8E result: 13.22% CAGR, Sharpe 2.61, Max DD -8.42%, positive 9/10 years
# Capital efficiency: 3% of total NAV caps naturally as positions accumulate;
# min 30-day hold gives predictable minimum capital commitment per trade.
ALLOCATION_METHOD   = "total_nav_pct"  # winner from §8E
TOTAL_NAV_PCT       = 0.03              # 3% of total portfolio NAV per trade
FIXED_TRADE_AMOUNT  = 100               # used if method == "fixed"
MAX_CONCURRENT      = 15                # used if method == "max_concurrent"
FREE_CAP_PCT        = 0.05              # used if method == "free_cap_frac"

SAVE_NAME           = "congress_trade_for_trade"


## 2. Load Data

In [ ]:
with open(DATA_PKL, "rb") as f:
    data = pickle.load(f)

final_df  = data["congresist_df_dict"]
stock_dfs = data["stock_dict"]
final_df['Ticker'] = final_df['Ticker'].replace('FB', 'META')

print(f"Trades: {len(final_df):,}  |  Tickers: {len(stock_dfs):,}  |  Politicians: {final_df['Name'].nunique()}")

## 3. Build Trade Pairs (Winner Config)

In [ ]:
# Apply Quiver delay cap
df = final_df.copy()
df['Filed']              = pd.to_datetime(df['Filed'])
df['Quiver_Upload_Time'] = pd.to_datetime(df['Quiver_Upload_Time'])
df['Filed_plus_cap']     = df['Filed'] + pd.Timedelta(days=QUIVER_DELAY_CAP)
df['Entry_Date']         = df[['Quiver_Upload_Time','Filed_plus_cap']].min(axis=1).dt.normalize()

# Purchases only
if PURCHASES_ONLY:
    df = df[df['Transaction'] == 'Purchase']
    print(f"Purchase disclosures: {len(df):,}")

In [ ]:
# Helper functions
def get_price(stock_dfs, ticker, date, prefer='close', direction='after'):
    if ticker not in stock_dfs: return np.nan
    s = stock_dfs[ticker].copy()
    s['timestamp'] = pd.to_datetime(s['timestamp']).dt.tz_localize(None).dt.normalize()
    s = s.sort_values('timestamp')
    if direction == 'after':
        rows = s[s['timestamp'] >= date]
    else:
        rows = s[s['timestamp'] <= date]
    return rows[prefer].iloc[0 if direction == 'after' else -1] if not rows.empty else np.nan

print("Building buy-sell trade pairs...")
pairs = []

for (politician, ticker), grp in df.groupby(['Name', 'Ticker']):
    buys  = grp.sort_values('Entry_Date')
    sales = final_df[
        (final_df['Name'] == politician) &
        (final_df['Ticker']     == ticker) &
        (final_df['Transaction'] == 'Sale')
    ].copy()
    if not sales.empty:
        sales['Filed'] = pd.to_datetime(sales['Filed'])
        sales['Entry_Date'] = pd.to_datetime(sales['Quiver_Upload_Time']).dt.normalize()

    for _, buy in buys.iterrows():
        buy_date = buy['Entry_Date']
        cap_date = buy_date + pd.Timedelta(days=HOLDING_CAP_DAYS)

        if not sales.empty:
            next_sale = sales[sales['Entry_Date'] > buy_date]
            if not next_sale.empty:
                raw_sell = next_sale.iloc[0]['Entry_Date']
                sell_date   = raw_sell if raw_sell <= cap_date else cap_date
                exit_reason = 'sale' if raw_sell <= cap_date else 'cap'
            else:
                sell_date, exit_reason = cap_date, 'cap'
        else:
            sell_date, exit_reason = cap_date, 'cap'

        bp = get_price(stock_dfs, ticker, buy_date, 'close', 'after')
        sp = get_price(stock_dfs, ticker, sell_date, 'close', 'before')
        ret = sp / bp - 1 if (bp and sp and bp > 0) else np.nan

        pairs.append({
            'Politician': politician, 'Ticker': ticker,
            'BuyDate': buy_date, 'SellDate': sell_date,
            'BuyPrice': bp, 'SellPrice': sp, 'Return': ret,
            'HoldingDays': (sell_date - buy_date).days,
            'ExitReason': exit_reason,
        })

pairs_df = pd.DataFrame(pairs)
print(f"Trade pairs: {len(pairs_df):,}  (sale: {(pairs_df['ExitReason']=='sale').sum():,}, cap: {(pairs_df['ExitReason']=='cap').sum():,})")
print(f"Mean return: {pairs_df['Return'].mean():.2%}  |  Missing prices: {pairs_df['Return'].isna().sum():,}")

In [ ]:
# Apply politician filter (winner config)
clean = pairs_df.dropna(subset=['Return']).copy()
clean['BuyDate'] = pd.to_datetime(clean['BuyDate'])

pol_stats = clean.groupby('Politician').apply(lambda g: pd.Series({
    'n_trades': len(g), 'years': g['BuyDate'].dt.year.nunique(),
    'win_rate': (g['Return'] > 0).mean(), 'mean_ret': g['Return'].mean(),
})).reset_index()

qualified_pols = pol_stats[
    (pol_stats['n_trades'] >= MIN_TRANSACTIONS) &
    (pol_stats['years']    >= MIN_YEARS_ACTIVE) &
    (pol_stats['win_rate'] >= MIN_WIN_RATE)
]['Politician']

dataset = clean[clean['Politician'].isin(qualified_pols)].sort_values('BuyDate').reset_index(drop=True)
print(f"Qualified politicians: {dataset['Politician'].nunique()} | Trades: {len(dataset):,}")
print(f"Mean return: {dataset['Return'].mean():.2%}  |  Win rate: {(dataset['Return']>0).mean():.2%}")

## 4. Statistical Significance

In [ ]:
trades_sig = dataset[['Return']].copy()
trades_sig['net_pnl']       = trades_sig['Return'] * FIXED_TRADE_AMOUNT
trades_sig['equity_before'] = INITIAL_CAPITAL

report = full_significance_report(trades_sig, strategy_name="Congress Trade-for-Trade D252")
print_significance_report(report)

## 5. Capital Simulation (Winner Allocation)

In [ ]:
def run_sim_winner(df, capital, method, fixed_amt=100, max_pos=15, free_pct=0.05,
                   kelly_frac=0.05, total_nav_pct=0.03, min_hold_days=30):
    """Run the winner allocation method. Set method to the V2_03 §8E winner."""
    import numpy as np
    df = df.sort_values('BuyDate').reset_index(drop=True)
    port, eq = [], []
    cap = capital

    for _, r in df.iterrows():
        buy, sell, ret = r['BuyDate'], r['SellDate'], r['Return']
        remaining = []
        for p in port:
            actual_close = max(p['sell'], p['min_sell'])  # enforce min hold
            if actual_close <= buy:
                cap += p['amt'] * (1 + p['ret'])
            else:
                remaining.append(p)
        port = remaining

        if method == 'fixed':
            amount = fixed_amt
        elif method == 'half_kelly':
            amount = cap * kelly_frac
        elif method == 'max_concurrent':
            if len(port) >= max_pos:
                eq.append((buy, cap + sum(p['amt'] for p in port)))
                continue
            pv = cap + sum(p['amt'] for p in port)
            amount = min(pv / max_pos, cap)
        elif method == 'free_cap_frac':
            amount = cap * free_pct
        elif method == 'total_nav_pct':  # winner E: % of total NAV, min hold
            total_nav = cap + sum(p['amt'] for p in port)
            amount = min(cap, total_nav * total_nav_pct)
        else:
            amount = fixed_amt

        min_sell_date = pd.Timestamp(buy) + pd.Timedelta(days=min_hold_days)
        if amount >= 1 and cap >= amount:
            cap -= amount
            port.append({'sell': sell, 'min_sell': min_sell_date, 'amt': amount, 'ret': ret})
        eq.append((buy, cap + sum(p['amt'] for p in port)))

    for p in port: cap += p['amt'] * (1 + p['ret'])

    equity = pd.DataFrame(eq, columns=['d','e']).sort_values('d').drop_duplicates('d').set_index('d')
    equity.index = pd.to_datetime(equity.index)
    days = (equity.index[-1] - equity.index[0]).days
    ann  = (cap / capital) ** (365 / days) - 1
    dr   = equity['e'].resample('D').last().ffill().pct_change().dropna()
    sh   = dr.mean() / dr.std() * np.sqrt(252) if dr.std() > 0 else 0
    peak = equity['e'].cummax()
    mdd  = ((equity['e'] - peak) / peak).min()
    per_year = dict(zip(equity['e'].resample('YE').last().index.year,
                        equity['e'].resample('YE').last().values /
                        equity['e'].resample('YE').last().shift(1).fillna(capital).values - 1))

    return equity['e'], cap, ann, sh, mdd, per_year

# Compute kelly fraction from data
wr = (dataset['Return'] > 0).mean()
b  = dataset[dataset['Return']>0]['Return'].mean() / abs(dataset[dataset['Return']<0]['Return'].mean())
kelly_half = max(0.01, (wr * b - (1-wr)) / b / 2)

equity, final_cap, ann, sh, mdd, per_year = run_sim_winner(
    dataset, INITIAL_CAPITAL, ALLOCATION_METHOD,
    fixed_amt=FIXED_TRADE_AMOUNT, max_pos=MAX_CONCURRENT,
    free_pct=FREE_CAP_PCT, kelly_frac=kelly_half,
    total_nav_pct=TOTAL_NAV_PCT, min_hold_days=MIN_HOLD_DAYS
)

print(f'Allocation method:  {ALLOCATION_METHOD}')
print(f'Initial capital:    ${INITIAL_CAPITAL:,.2f}')
print(f'Final capital:      ${final_cap:,.2f}')
print(f'Total return:       {final_cap/INITIAL_CAPITAL-1:+.2%}')
print(f'Annualized return:  {ann:+.2%}')
print(f'Sharpe ratio:       {sh:.2f}')
print(f'Max drawdown:       {mdd:.2%}')
print(f'N trades:           {len(dataset):,}')
print()
print('Per-year returns:')
for yr, r in per_year.items():
    print(f'  {yr}: {r:+.2%}')


## 6. Equity Curve + Drawdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})

ax1.plot(equity.index, equity.values, linewidth=1.5, color='seagreen',
         label=f"{ALLOCATION_METHOD} — {ann:+.1%} ann | Sh {sh:.2f} | DD {mdd:.1%}")
ax1.axhline(INITIAL_CAPITAL, color='black', linestyle=':', alpha=0.4)
ax1.set_ylabel('Portfolio Value ($)')
ax1.set_title('Congress Trade-for-Trade D252 — Portfolio B')
ax1.legend()
ax1.grid(alpha=0.3)

dd_s = (equity - equity.cummax()) / equity.cummax()
ax2.fill_between(dd_s.index, dd_s.values, 0, alpha=0.5, color='red')
ax2.set_ylabel('Drawdown')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Save Trades

In [ ]:
import os
os.makedirs("results", exist_ok=True)

trades_out = dataset.copy().rename(columns={
    'BuyDate':'entry_time', 'SellDate':'exit_time', 'Ticker':'instrument',
    'Return':'pct_return_gross', 'BuyPrice':'entry_price', 'SellPrice':'exit_price',
    'ExitReason':'exit_reason',
})
trades_out['direction']  = 'long'
trades_out['stop_price'] = np.nan

std_cols = ['entry_time','exit_time','direction','instrument',
            'entry_price','exit_price','pct_return_gross','exit_reason','stop_price']
path = f"results/{SAVE_NAME}_trades.csv"
trades_out[std_cols].to_csv(path, index=False)
print(f"Saved {len(trades_out):,} trades → {path}")

## 8. Conclusions

**Strategy: Congress Trade-for-Trade D252 — Portfolio B**

| Metric | Value |
|---|---|
| Allocation method | see CONFIGURATION |
| Annualized return | see §5 output |
| Sharpe ratio | see §5 output |
| Max drawdown | see §5 output |
| Positive years | see §5 output |
| N trades (backtest) | see §5 output |

**Go/No-go criteria:**
- [ ] Sharpe > 0.7
- [ ] Annualized return > 10%
- [ ] Statistical significance: 2/3 tests pass (see §4)
- [ ] Max drawdown < 20%

**To update allocation method:** Run `V2_03_trade_for_trade.ipynb` §8E, find winner, then update `ALLOCATION_METHOD` in §1 config of this notebook.

**Next:** `congress_trade_for_trade_Implementation.ipynb` — poll Quiver for new purchases, manage open positions, exit on Sale alert or cap.

In [ ]:
# ── Save outputs for Portfolio_B_Analysis.ipynb ──────────────────────────────
import os, json as _json

SAVE_NAME = 'congress_trade_for_trade'
os.makedirs(f'results/{SAVE_NAME}_daily_equity', exist_ok=True)

# equity = the equity pd.Series returned by run_sim_winner()
daily = equity.resample('B').last().ffill()
daily.rename('equity').to_frame().rename_axis('date').to_csv(
    f'results/{SAVE_NAME}_daily_equity/total_nav_3pct_d180_min30d_1x.csv'
)

final_cap = daily.iloc[-1]
days = (daily.index[-1] - daily.index[0]).days
cagr = (final_cap / INITIAL_CAPITAL) ** (365 / days) - 1
dr = daily.pct_change().dropna()
sharpe = dr.mean() / dr.std() * (252 ** 0.5) if dr.std() > 0 else 0
max_dd = ((daily - daily.cummax()) / daily.cummax()).min()

impl = {
    'total_nav_3pct_d180_min30d_1x': {
        'total_return': round((final_cap / INITIAL_CAPITAL - 1) * 100, 1),
        'cagr':   round(cagr * 100, 2),
        'sharpe': round(sharpe, 2),
        'max_dd': round(max_dd * 100, 1),
        'label':  '3% total NAV, D180 cap, min 30d hold (1x)',
    }
}
with open(f'results/{SAVE_NAME}_implementations.json', 'w') as f:
    _json.dump(impl, f, indent=2)

print(f'Saved results/{SAVE_NAME}_implementations.json')
print(f'Saved results/{SAVE_NAME}_daily_equity/total_nav_3pct_d180_min30d_1x.csv')
print(f'CAGR: {cagr:.2%}  Sharpe: {sharpe:.2f}  MaxDD: {max_dd:.2%}')
